# SAM3 trực tiếp từ Hugging Face trên Colab

Notebook dùng Google Drive làm cache. `facebook/sam3` chỉ tải một lần, sau đó dùng lại. Chọn bất kỳ task nào: `easy_semantic`, `medium_instance`, `hard_panoptic`, `cp1_holes`, `cp2_slice`, `cp3_thin`, `cp4_curb`, `cp5_occlusion`, `cp6_coverage`. Notebook tự chọn format: semantic dùng Segmentation mask 1.1; instance/panoptic dùng COCO 1.0.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
HF_DIR = Path('/content/drive/MyDrive/huggingface')
os.environ['HF_HOME'] = str(HF_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_DIR / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(HF_DIR / 'transformers')
HF_DIR.mkdir(parents=True, exist_ok=True)
print('Cache:', HF_DIR)

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub pycocotools

In [ ]:
from huggingface_hub import login
login()
# Dán Hugging Face token khi được hỏi. Tài khoản phải được cấp quyền facebook/sam3.

In [ ]:
# Chỉ đặt True khi lần chạy trước báo 'SafetensorError: header too large'.
# Sau khi tải thành công một lần, đổi lại thành False để giữ cache.
RESET_SAM3_CACHE = True

import shutil
from huggingface_hub import snapshot_download

model_cache = HF_DIR / 'hub' / 'models--facebook--sam3'
if RESET_SAM3_CACHE and model_cache.exists():
    shutil.rmtree(model_cache)
    print('Đã xóa cache SAM3 cũ.')

snapshot_download(
    repo_id='facebook/sam3',
    cache_dir=str(HF_DIR / 'hub'),
    resume_download=True,
)
print('Đã tải/kiểm tra SAM3 trong Google Drive:', model_cache)

In [ ]:
REPO_URL = 'https://github.com/<username>/<repo>.git'
REPO_DIR = '/content/image-segmentation'

import subprocess
from pathlib import Path
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

In [ ]:
TASK = 'medium_instance'

TASK_DIRS = {
    'easy_semantic': 'data/tiers/easy_semantic',
    'medium_instance': 'data/tiers/medium_instance',
    'hard_panoptic': 'data/tiers/hard_panoptic',
    'cp1_holes': 'data/checkpoints/cp1_holes',
    'cp2_slice': 'data/checkpoints/cp2_slice',
    'cp3_thin': 'data/checkpoints/cp3_thin',
    'cp4_curb': 'data/checkpoints/cp4_curb',
    'cp5_occlusion': 'data/checkpoints/cp5_occlusion',
    'cp6_coverage': 'data/checkpoints/cp6_coverage',
}
assert TASK in TASK_DIRS, f'Chọn một trong: {list(TASK_DIRS)}'
task_dir = Path(TASK_DIRS[TASK])
images = task_dir / 'images'
classes = task_dir / 'classes.json'
output = Path('submissions') / f'{TASK}.zip'
print(TASK, images, classes, output)

In [ ]:
import subprocess, json
info = json.loads(classes.read_text())
cmd = ['python', 'scripts/sam3_hf_coco.py', '--images', str(images), '--classes', str(classes), '--output', str(output), '--device', 'cuda', '--threshold', '0.25', '--format', 'auto']
if info.get('type') == 'panoptic':
    cmd.append('--panoptic')
print('Đang chạy:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
!python scripts/inspect_submissions.py --task "$TASK" --dir submissions

In [ ]:
from google.colab import files
files.download(str(output))